In [14]:
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
import scipy
import plotly.express as px
import requests
import io
import streamlit as st

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 1000) # Display more row of your dataframe
pd.options.display.float_format = '{:,}'.format # Separate number with comma


In [15]:
BASE_couverture_ciblée = "https://data.corsica/data-fair/api/v1/datasets/dispositif-de-couverture-ciblee-en-corse-points-dinteret/"

df = pd.read_csv(f"{BASE_couverture_ciblée}/lines?format=csv")
colonnes =[]
for col in df.columns:
    colonnes.append(col)

colonnes
params = {
    "format": "csv",
    "select": ",".join(colonnes),
    "size": 10000,   # maximum par page
}

In [17]:
url = f"{BASE_couverture_ciblée}/lines"
params = {
    "format": "json",
    "select": ",".join(colonnes),
    "size": 10000,   # maximum par page
}
lignes=[]

while url:
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    lignes.extend(data["results"])
    url = data.get("next")
    params = None   # l'URL "next" contient déjà les paramètres

df = pd.DataFrame(lignes)

colonnes_a_supprimer = [
    "20801",
    "20810",
    "20815",
    "20820",
    "id_point",
    "nb_sites_dossier",
    "origine_zone",
    "origine_coordonnées",
    "num_zone_arrete",
    "nom_point_arrete",
    "lien_arrete",
    "_i",
    "_rand",
    "_score",
    "_id",
    "numero_site_physique",
    "_identifiant_site",
]

df = df.drop(columns=colonnes_a_supprimer, errors="ignore")
df.to_csv("csv/couverture_reseau.csv", index=False)